# Michigan Traders — Module 1
# NumPy for Quants  ·  *interactive workbook*

**Series:** MAT Education · Foundations
**Level:** Intermediate (you know Python; here we build the numerical muscle quant work runs on)
**Format:** **Interactive** — every idea gets an explanation, a worked example, then a **`# TODO` for you**.

---

Almost every quantitative idea you'll meet this year — returns, volatility, Sharpe ratios, covariance matrices, Monte-Carlo simulation, the math inside an ML model — is, under the hood, **array arithmetic**. **NumPy** (Numerical Python) is the library that makes that arithmetic fast, concise, and correct. It gives you a new data type — the **array** (`ndarray`) — and a huge toolbox of operations on it.

We teach NumPy *through finance*: every example uses prices, returns, or portfolios. Take it slowly — read the explanation, run the example, understand the output, *then* do the exercise.

## How to use this notebook

Runs in a normal Jupyter kernel — no QuantConnect needed:

```bash
pip install numpy matplotlib
```

For each topic you'll see:

1. **An explanation** of the idea.
2. **A worked example** (already run — read the code *and* its output).
3. **✏️ Your turn** — a cell with `# TODO`. Replace the `None`/blank with your code.
4. **A self-check cell** — run it. **`✅ Correct!`** = you nailed it; an `AssertionError` tells you what's off.

> **Stuck?** The full worked answer key is the companion notebook **`01_NumPy_for_Quants_SOLUTIONS.ipynb`** — but wrestle with the TODO first. That struggle is the learning.

Run the setup cell below, then go top to bottom.

### Setup

By convention NumPy is always imported as `np` — you'll see `np.` everywhere. We also create a **seeded random number generator** so that every "random" number below is reproducible: you'll get the *same* values shown in the comments, which makes it possible to check your answers. (More on the generator in §2.)

In [1]:
import numpy as np

rng = np.random.default_rng(42)   # a random generator seeded with 42 -> reproducible
np.set_printoptions(precision=4, suppress=True)   # print arrays with 4 decimals, no sci-notation
print("NumPy version:", np.__version__)

NumPy version: 2.5.0


## 1. Why NumPy? Arrays vs. Python lists

A Python `list` can hold anything, but doing math on it means looping element by element — slow, and verbose to write. A NumPy **array** stores one data type (e.g. all floats) in a single contiguous block of memory, so NumPy can run operations as compiled C loops over that block.

Two ideas to internalize right now:

- **Element-wise by default** — arithmetic on an array applies to *every element at once*. No loop needed.
- **Vectorization** — expressing a computation as whole-array operations (instead of a Python `for` loop) is both shorter to write and dramatically faster to run.

Let's see "element-wise" first.

In [2]:
a = np.array([1, 2, 3, 4])

print("a        =", a)
print("a + 10   =", a + 10)    # adds 10 to EVERY element
print("a * 2    =", a * 2)     # doubles every element
print("a ** 2   =", a ** 2)    # squares every element

b = np.array([10, 20, 30, 40])
print("a + b    =", a + b)     # element-by-element: [1+10, 2+20, 3+30, 4+40]

a        = [1 2 3 4]
a + 10   = [11 12 13 14]
a * 2    = [2 4 6 8]
a ** 2   = [ 1  4  9 16]
a + b    = [11 22 33 44]


Notice there is **no `for` loop** anywhere above — yet every element got transformed. With a Python list, `a + 10` would be an error and you'd have to write `[x + 10 for x in a]`.

**Now, why does this matter for speed?** The next cell runs a little experiment: it squares **one million numbers two different ways** and times each.

- **Way 1 — a plain Python loop** (a list comprehension `[x * x for x in py_list]`): the interpreter steps through all one million elements one at a time.
- **Way 2 — NumPy vectorized** (`np_arr * np_arr`): NumPy multiplies the whole array against itself in one compiled C operation.

Both produce the *same* result; we only compare how long each takes.

In [3]:
import time

n = 1_000_000
py_list = list(range(n))     # a Python list of 1,000,000 ints
np_arr  = np.arange(n)       # the same numbers as a NumPy array

# Way 1: square every element with a Python loop, and time it.
t0 = time.perf_counter()
loop_result = [x * x for x in py_list]
t_loop = time.perf_counter() - t0

# Way 2: square every element with NumPy, and time it.
t0 = time.perf_counter()
numpy_result = np_arr * np_arr
t_numpy = time.perf_counter() - t0

print(f"Python loop : {t_loop * 1000:7.1f} ms")
print(f"NumPy       : {t_numpy * 1000:7.1f} ms")
print(f"NumPy is about {t_loop / t_numpy:.0f}x faster (the exact number varies by machine)")

Python loop :    25.4 ms
NumPy       :     1.8 ms
NumPy is about 14x faster (the exact number varies by machine)


NumPy wins by a wide margin — often 10–100×. The reason: the Python loop pays interpreter overhead on every one of the million elements, while NumPy hands the whole job to compiled C code working over contiguous memory.

**The takeaway you'll use all year:** in quant code, reach for a vectorized array operation *before* you write a Python loop. A backtest that loops in Python over millions of bars can be minutes slower than the same logic vectorized.

## 2. Creating arrays

Before you can compute anything you need arrays to compute *with*. NumPy gives you several constructors — we'll go through them one at a time, because you'll reach for different ones in different situations.

**`np.array(...)`** — the most basic: turn an existing Python list (or list of lists) into an array. This is what you'll use when you already have the numbers in hand.

In [4]:
prices = np.array([100.0, 101.5, 99.8, 102.3, 103.1])
print("prices       =", prices)
print("type         =", type(prices))
print("element type =", prices.dtype)   # float64 - all elements share one dtype

prices       = [100.  101.5  99.8 102.3 103.1]
type         = <class 'numpy.ndarray'>
element type = float64


**`np.arange(start, stop, step)`** — like Python's built-in `range`, but it returns an **array**. The `stop` value is *excluded*. Handy for making an index of day numbers, or any evenly-stepped sequence.

In [5]:
print("np.arange(5)        =", np.arange(5))         # 0..4 (stop excluded)
print("np.arange(0, 10, 2) =", np.arange(0, 10, 2))  # start 0, stop 10, step 2
print("np.arange(2021, 2025) =", np.arange(2021, 2025))

np.arange(5)        = [0 1 2 3 4]
np.arange(0, 10, 2) = [0 2 4 6 8]
np.arange(2021, 2025) = [2021 2022 2023 2024]


**`np.linspace(start, stop, num)`** — give it a start, a stop, and **how many points** you want, and it spreads them *evenly* between the two (including both endpoints). Use this when you care about the *count* of points, not the step size — e.g. sweeping a portfolio weight from 0% to 100% in 5 steps.

Contrast with `arange`: with `arange` you pick the **step**; with `linspace` you pick the **number of points** (and endpoints are included).

In [6]:
print("linspace(0, 1, 5) =", np.linspace(0, 1, 5))    # 5 points from 0 to 1 inclusive
print("linspace(0, 1, 11) =", np.linspace(0, 1, 11))   # 11 points -> step 0.1

linspace(0, 1, 5) = [0.   0.25 0.5  0.75 1.  ]
linspace(0, 1, 11) = [0.  0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1. ]


**`np.zeros(n)`, `np.ones(n)`, `np.full(n, value)`** — make an array of a given size pre-filled with `0`, `1`, or any constant. Two common uses: (a) **pre-allocating** an array you'll fill in later, and (b) making a **constant vector** — e.g. an equal-weight portfolio where every weight is the same number.

In [7]:
print("zeros(4)        =", np.zeros(4))
print("ones(3)         =", np.ones(3))
print("full(3, 0.05)   =", np.full(3, 0.05))      # three weights of 5% each
print("ones(4) / 4     =", np.ones(4) / 4)        # another way to get equal weights (0.25 each)

zeros(4)        = [0. 0. 0. 0.]
ones(3)         = [1. 1. 1.]
full(3, 0.05)   = [0.05 0.05 0.05]
ones(4) / 4     = [0.25 0.25 0.25 0.25]


**2-D arrays (matrices).** Pass a *list of lists* to `np.array` and you get a 2-D array. Throughout quant work the mental model is: **rows = time (days), columns = assets**. So `data[2, 1]` is the value at day 2, asset 1.

In [8]:
data = np.array([[100.0, 50.0],     # day 0:  asset0=100, asset1=50
                 [101.0, 49.5],     # day 1
                 [102.5, 51.0]])    # day 2
print("the matrix:\n", data)
print("rows, cols =", data.shape)   # (3 days, 2 assets)

the matrix:
 [[100.   50. ]
 [101.   49.5]
 [102.5  51. ]]
rows, cols = (3, 2)


**Random arrays.** For simulations and test data you'll generate random numbers. Always go through a **seeded generator** (`rng = np.random.default_rng(seed)`) rather than the old global `np.random.*` functions — seeding makes results reproducible, which is essential for research you can trust and for checking answers. `rng.normal(mean, std, size)` draws from a normal (bell-curve) distribution — perfect for simulating returns.

In [9]:
rng_demo = np.random.default_rng(0)               # seed 0
print("4 normal draws (mean 0, std 1):", rng_demo.normal(0, 1, size=4))
print("simulated daily returns:       ", rng_demo.normal(0.0005, 0.01, size=5))   # ~0.05% mean, 1% vol

4 normal draws (mean 0, std 1): [ 0.1257 -0.1321  0.6404  0.1049]
simulated daily returns:        [-0.0049  0.0041  0.0135  0.01   -0.0065]


### ✏️ Your turn — equal-weight portfolio

Build `weights`, an **equal-weight** vector over `n_assets = 5` (each weight = 1/5), using a NumPy call — not a Python list. It should sum to 1. (Hint: `np.full` or `np.ones`.)

In [ ]:
n_assets = 5
weights = None   # TODO: an equal-weight NumPy vector of length n_assets

In [ ]:
assert weights is not None, "Replace None with your code."
assert weights.shape == (5,), f"Expected shape (5,), got {weights.shape}"
assert np.isclose(weights.sum(), 1.0), "Weights must sum to 1."
assert np.allclose(weights, 0.2), "Each weight should be 0.2."
print("✅ Correct!", weights)

## 3. Inspecting and reshaping arrays

Once you have an array, four attributes tell you what it is. **Printing `.shape` is the single most useful habit in NumPy** — most bugs are really shape mismatches.

- `.shape` — the dimensions, e.g. `(252, 4)` = 252 rows × 4 columns
- `.ndim` — number of dimensions (1 for a vector, 2 for a matrix)
- `.size` — total number of elements
- `.dtype` — the element type (e.g. `float64`)

In [12]:
returns_matrix = rng.normal(0, 0.01, size=(252, 4))   # 252 days x 4 assets

print(".shape =", returns_matrix.shape)   # (252, 4)
print(".ndim  =", returns_matrix.ndim)    # 2
print(".size  =", returns_matrix.size)    # 252 * 4 = 1008
print(".dtype =", returns_matrix.dtype)   # float64

.shape = (252, 4)
.ndim  = 2
.size  = 1008
.dtype = float64


**`reshape`** rearranges the same data into a new shape (the total number of elements must match). **`astype`** converts the element type.

In [13]:
flat = np.arange(12)            # [0, 1, ..., 11], shape (12,)
print("reshaped to 3x4:\n", flat.reshape(3, 4))
print("\nfloats -> ints (truncates toward zero):", np.array([1.9, 2.1, 3.7]).astype(int))

reshaped to 3x4:
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]

floats -> ints (truncates toward zero): [1 2 3]


## 4. Indexing & slicing — and the view-vs-copy trap

Indexing a 1-D array works just like a Python list: `a[0]` is the first element, `a[-1]` the last, `a[1:4]` a slice.

In [14]:
prices = np.array([100., 101., 102., 103., 104., 105.])
print("first element a[0]   :", prices[0])
print("last element  a[-1]  :", prices[-1])
print("slice a[1:4]         :", prices[1:4])    # positions 1,2,3 (4 excluded)
print("every other a[::2]   :", prices[::2])

first element a[0]   : 100.0
last element  a[-1]  : 105.0
slice a[1:4]         : [101. 102. 103.]
every other a[::2]   : [100. 102. 104.]


For a **2-D array** you index with `[row, column]`. A colon `:` means "everything along this axis" — so `data[:, 2]` is "every row, column 2" (an entire asset's price series), and `data[1, :]` is "row 1, every column" (all assets on one day).

In [15]:
data = np.arange(12).reshape(4, 3)   # 4 days, 3 assets
print("matrix:\n", data)
print("\nelement [0, 1] (day 0, asset 1):", data[0, 1])
print("column 2  data[:, 2] (asset 2, all days):", data[:, 2])
print("row 1     data[1, :] (day 1, all assets):", data[1, :])

matrix:
 [[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]]

element [0, 1] (day 0, asset 1): 1
column 2  data[:, 2] (asset 2, all days): [ 2  5  8 11]
row 1     data[1, :] (day 1, all assets): [3 4 5]


**The trap that bites everyone once.** A basic slice returns a **view** — a window onto the *same* memory, not a fresh copy. Writing into the view changes the original array. When you need an independent copy, call `.copy()`.

In [16]:
p = np.array([100., 101., 102., 103.])
window = p[:2]          # a VIEW of the first two elements
window[0] = -999        # writing into the view...
print("original after writing into the view:", p)   # ...changed `p` too!

p = np.array([100., 101., 102., 103.])
safe = p[:2].copy()     # an independent COPY
safe[0] = -999
print("original after writing into a .copy():", p)   # untouched

original after writing into the view: [-999.  101.  102.  103.]
original after writing into a .copy(): [100. 101. 102. 103.]


### ✏️ Your turn — grab a price series

From `data = np.arange(12).reshape(4, 3)` (rows = days, cols = assets), extract **asset index 1's** full series across all days into `asset1`. (Hint: every row, one column.)

In [ ]:
data = np.arange(12).reshape(4, 3)
asset1 = None   # TODO: all days for the asset at column index 1

In [ ]:
assert asset1 is not None, "Replace None with your code."
assert np.array_equal(asset1, np.array([1, 4, 7, 10])), f"Got {asset1}"
print("✅ Correct! asset 1 series:", asset1)

## 5. Boolean masks — asking questions of data without loops

A **comparison** on an array produces a **boolean array** (a *mask*) — `True`/`False` for each element. You then use that mask to select, count, or transform elements. This is how you answer "which days were down?", "how many moves exceeded 2%?", etc.

In [19]:
r = np.array([0.012, -0.004, 0.0, 0.021, -0.018, 0.007])

mask = r < 0                 # a boolean array: True where the return is negative
print("returns:", r)
print("mask (r < 0):", mask)

returns: [ 0.012 -0.004  0.     0.021 -0.018  0.007]
mask (r < 0): [False  True False False  True False]


**Selecting** with a mask returns only the elements where it's `True`. And because `True` counts as `1` and `False` as `0`, you can `.sum()` a mask to **count** matches, or `.mean()` it to get the **fraction** that are `True`.

In [20]:
print("the negative returns      :", r[mask])     # select where mask is True
print("number of down days (sum) :", mask.sum())  # True==1, so sum = count
print("fraction down (mean)      :", mask.mean()) # fraction of days that were down

the negative returns      : [-0.004 -0.018]
number of down days (sum) : 2
fraction down (mean)      : 0.3333333333333333


**Combining conditions:** use `&` (and) / `|` (or), and **wrap each condition in parentheses** — operator precedence makes the parentheses mandatory, or you'll get an error.

In [21]:
big_moves = r[(r > 0.01) | (r < -0.01)]    # moves bigger than +/-1%
print("big moves:", big_moves)

big moves: [ 0.012  0.021 -0.018]


**`np.where(condition, x, y)`** is a vectorized if/else: wherever the condition is `True` it takes `x`, otherwise `y`. A natural use is turning returns into a simple long/short signal.

In [22]:
signal = np.where(r > 0, 1, -1)   # +1 on up days, -1 on down days
print("signal:", signal)

signal: [ 1 -1 -1  1 -1  1]


### ✏️ Your turn — win rate

Given `rets` below, compute `win_rate` — the **fraction of strictly positive days**. (Hint: build the mask `rets > 0`, then `.mean()` it.)

In [ ]:
rets = np.array([0.01, -0.02, 0.005, -0.001, 0.03, -0.04, 0.012])
win_rate = None   # TODO: fraction of days with rets > 0

In [ ]:
assert win_rate is not None, "Replace None with your code."
assert np.isclose(win_rate, 4 / 7), f"Expected 4/7 = {4/7:.4f}, got {win_rate}"
print(f"✅ Correct! win rate = {win_rate:.1%}")

## 6. Vectorized math & broadcasting — prices → returns

Now the single most common transform in trading: turning a **price series** into a **return series**.

The trick is to line up each price with the one before it using two slices:
- `prices[1:]`  — every price *except the first* (today)
- `prices[:-1]` — every price *except the last* (yesterday)

Dividing them element-wise pairs each day with the prior day. A **simple return** is `P_today / P_yesterday - 1`.

In [25]:
prices = np.array([100., 102., 101., 105., 107., 103.])

print("prices[1:]  (today)    :", prices[1:])
print("prices[:-1] (yesterday):", prices[:-1])

simple_returns = prices[1:] / prices[:-1] - 1
print("simple returns         :", simple_returns)

prices[1:]  (today)    : [102. 101. 105. 107. 103.]
prices[:-1] (yesterday): [100. 102. 101. 105. 107.]
simple returns         : [ 0.02   -0.0098  0.0396  0.019  -0.0374]


A **log return** is `ln(P_today / P_yesterday)`. `np.log` applies the natural log element-wise, and `np.diff` takes successive differences — so `np.diff(np.log(prices))` is a compact way to get log returns. (Log returns are *additive over time*, which makes a lot of math cleaner; you'll use both kinds.)

In [26]:
log_returns = np.diff(np.log(prices))
print("log returns:", log_returns)

log returns: [ 0.0198 -0.0099  0.0388  0.0189 -0.0381]


**Broadcasting** is NumPy's rule for combining arrays of *different but compatible* shapes: it "stretches" the smaller one to fit. The classic use is normalizing a matrix column-by-column. Below, `R` is `(252, 4)`, while `R.mean(axis=0)` and `R.std(axis=0)` are each `(4,)` — one value per column. NumPy broadcasts those per-column numbers across all 252 rows, so the whole matrix gets z-scored (each column centered to mean 0, scaled to std 1) in one line.

In [27]:
R = rng.normal(0, 0.01, size=(252, 4))
z = (R - R.mean(axis=0)) / R.std(axis=0)      # (252,4) combined with (4,) -> broadcast
print("each z-scored column has mean ~0:", z.mean(axis=0))
print("each z-scored column has std  ~1:", z.std(axis=0))

each z-scored column has mean ~0: [-0.  0. -0.  0.]
each z-scored column has std  ~1: [1. 1. 1. 1.]


### ✏️ Your turn — log returns

Compute the **log returns** of `prices` into `log_ret` (length 5). Recall: log return $= \ln(P_t / P_{t-1})$. `np.diff` and `np.log` are useful.

In [ ]:
prices = np.array([100., 102., 101., 105., 107., 103.])
log_ret = None   # TODO: the log returns of `prices`

In [ ]:
assert log_ret is not None, "Replace None with your code."
assert log_ret.shape == (5,), f"Expected shape (5,), got {log_ret.shape}"
assert np.allclose(log_ret[0], 0.0198, atol=1e-4), "First log return should be ~0.0198."
assert np.allclose(log_ret, np.diff(np.log(prices))), "Values don't match log returns."
print("✅ Correct!", np.round(log_ret, 4))

## 7. Aggregations and the `axis` argument

**Aggregations** (also called reductions) collapse many numbers into fewer: `sum`, `mean`, `std`, `min`, `max`. On a 1-D array they return a single number.

In [30]:
daily = np.array([0.01, -0.02, 0.015, 0.004, -0.008])
print("sum  :", daily.sum())
print("mean :", daily.mean())
print("std  :", daily.std())
print("min  :", daily.min(), " max:", daily.max())

sum  : 0.0009999999999999992
mean : 0.00019999999999999982
std  : 0.012687001221722964
min  : -0.02  max: 0.015


On a **2-D** array you must say *which direction* to collapse — that's the **`axis`** argument, and it's worth getting crystal clear:

- **`axis=0`** collapses **down the rows** → one result **per column** (per *asset*).
- **`axis=1`** collapses **across the columns** → one result **per row** (per *day*).
- **no axis** → collapse everything to a single scalar.

Mnemonic: `axis=0` is the *row* index, so it's the axis that *disappears* — you're left with columns.

In [31]:
R = rng.normal(0.0004, 0.01, size=(252, 4))   # 252 days x 4 assets

print("mean per ASSET, axis=0 ->", R.mean(axis=0))            # 4 numbers (one per column)
print("mean per DAY,   axis=1 -> first 5:", R.mean(axis=1)[:5])  # 252 numbers (one per row)
print("overall mean (no axis) ->", R.mean())                  # 1 number

mean per ASSET, axis=0 -> [0.0007 0.0007 0.0009 0.0006]
mean per DAY,   axis=1 -> first 5: [ 0.0013 -0.0014 -0.0117  0.0018  0.0032]
overall mean (no axis) -> 0.0007261754559047926


A daily mean/std isn't very intuitive, so we **annualize**: multiply a mean return by 252 (trading days/year), and multiply a volatility (std) by `√252`. (Variance scales with time, so volatility scales with its square root.)

In [32]:
ann_return = R.mean(axis=0) * 252
ann_vol    = R.std(axis=0) * np.sqrt(252)
print("annualized return per asset:", ann_return.round(3))
print("annualized vol    per asset:", ann_vol.round(3))

annualized return per asset: [0.169 0.187 0.232 0.144]
annualized vol    per asset: [0.158 0.151 0.165 0.173]


### ✏️ Your turn — annualized volatility

Given the returns matrix `R` below (252 days × 3 assets), compute `ann_vol` = the **annualized volatility per asset** (daily std × √252). Shape should be `(3,)` — so think about which `axis`.

In [ ]:
R = np.random.default_rng(1).normal(0, 0.01, size=(252, 3))
ann_vol = None   # TODO: annualized vol per asset

In [ ]:
assert ann_vol is not None, "Replace None with your code."
assert ann_vol.shape == (3,), f"Expected (3,), got {ann_vol.shape} - check your axis!"
assert np.allclose(ann_vol, R.std(axis=0) * np.sqrt(252)), "Values off - did you annualize with sqrt(252)?"
assert ((ann_vol > 0.10) & (ann_vol < 0.25)).all(), "Sanity: ~0.01 daily vol annualizes to ~0.16."
print("✅ Correct! annualized vol per asset:", np.round(ann_vol, 4))

## 8. Worked example: a strategy's performance stats

Time to combine everything. Given a price path, we'll compute the metrics that appear on *every* backtest report. Here's what each one means:

- **Annualized return** — average daily return scaled to a year.
- **Annualized volatility** — how much returns bounce around, scaled to a year (risk).
- **Sharpe ratio** — return per unit of risk (`return / volatility`, assuming a 0% risk-free rate). Higher is better; ~1 is decent, ~2 is excellent.
- **Equity curve** — growth of \$1 over time, via compounding: `(1 + r).cumprod()`.
- **Max drawdown** — the worst peak-to-trough drop in the equity curve (how much you'd have lost from a high).

First we simulate a 3-year daily price path (the simulation math is explained in §9 — for now just treat it as a realistic random price series), then compute the headline stats.

In [35]:
# Simulate ~3 years of daily prices (details of this formula are explained in the next section).
rng2 = np.random.default_rng(9)
n_days, mu, sigma, dt = 252 * 3, 0.08, 0.20, 1 / 252
daily_log = rng2.normal((mu - 0.5 * sigma**2) * dt, sigma * np.sqrt(dt), n_days)
px = 100 * np.exp(np.cumsum(daily_log))   # turn log returns into a price path

ret = px[1:] / px[:-1] - 1                # daily simple returns

ann_return = ret.mean() * 252
ann_vol    = ret.std() * np.sqrt(252)
sharpe     = ann_return / ann_vol
print(f"annualized return : {ann_return:6.2%}")
print(f"annualized vol    : {ann_vol:6.2%}")
print(f"Sharpe ratio      : {sharpe:6.2f}")

annualized return : 11.56%
annualized vol    : 19.95%
Sharpe ratio      :   0.58


Now the **equity curve** and **drawdown**. `np.cumprod(1 + ret)` compounds the returns into growth-of-\$1. For drawdown we need the *running peak* — the highest equity seen so far at each point — which is exactly what **`np.maximum.accumulate`** gives you in one vectorized call. Drawdown is then `equity / running_peak - 1` (always ≤ 0), and the worst one is its minimum.

In [36]:
equity = np.cumprod(1 + ret)               # growth of $1
running_peak = np.maximum.accumulate(equity)   # highest equity seen so far
drawdown = equity / running_peak - 1           # how far below the peak, at each point
print(f"final equity ($1 ->) : {equity[-1]:.2f}")
print(f"max drawdown         : {drawdown.min():.2%}")

final equity ($1 ->) : 1.33
max drawdown         : -15.93%


### ✏️ Your turn — max drawdown

Given the `equity` curve below (growth of \$1), compute `max_dd`, the **worst peak-to-trough drawdown** (a negative number). Use `np.maximum.accumulate(equity)` for the running peak, then take the minimum of `equity / running_peak - 1`.

In [ ]:
equity = np.array([1.0, 1.1, 1.05, 1.2, 0.9, 1.0, 1.3])
max_dd = None   # TODO: most negative value of (equity / running_peak - 1)

In [ ]:
assert max_dd is not None, "Replace None with your code."
assert np.isclose(max_dd, -0.25), f"Expected -0.25 (0.9 down from a 1.2 peak), got {max_dd}"
print(f"✅ Correct! max drawdown = {max_dd:.1%}")

## 9. Random numbers & Monte-Carlo simulation

A **Monte-Carlo simulation** answers "what's the *distribution* of possible outcomes?" by generating many random scenarios and looking at the spread. It's central to risk (Value-at-Risk), option pricing, and stress-testing.

We'll model prices with **Geometric Brownian Motion (GBM)** — the standard "random walk with drift" for asset prices. Intuition: each day the price gets multiplied by `exp(a small random number)`, where that random number has a slight upward **drift** (`mu`) and some **volatility** (`sigma`). You don't need to memorize the formula; just know it produces realistic-looking random price paths.

The key NumPy move: generate **all** the random shocks at once into a `(days, paths)` matrix, then `cumsum` down the days — no path-by-path Python loop.

In [39]:
rng3 = np.random.default_rng(123)
n_paths, n_days = 10_000, 252
S0, mu, sigma, dt = 100.0, 0.07, 0.25, 1 / 252

# One big matrix of daily log-shocks: shape (days, paths).
shocks = rng3.normal((mu - 0.5 * sigma**2) * dt, sigma * np.sqrt(dt), size=(n_days, n_paths))

# cumsum DOWN the days (axis=0), exponentiate -> 10,000 price paths at once.
paths = S0 * np.exp(np.cumsum(shocks, axis=0))   # shape (252, 10000)
terminal = paths[-1]                              # the final price of each path
print("simulated", n_paths, "one-year paths")
print("a few terminal prices:", terminal[:5].round(2))

simulated 10000 one-year paths
a few terminal prices: [148.37 126.83 111.23 104.36  85.38]


Now summarize the distribution of where prices ended up. **Percentiles** describe the spread, and a simple **Value-at-Risk** reads off the downside: the 5th percentile of returns is the loss you'd only exceed 5% of the time.

In [40]:
terminal_return = terminal / S0 - 1
print(f"mean terminal price : {terminal.mean():.2f}")
print(f"5th percentile      : {np.percentile(terminal, 5):.2f}")
print(f"95th percentile     : {np.percentile(terminal, 95):.2f}")
print(f"95% 1-year VaR (loss): {-np.percentile(terminal_return, 5):.2%}")

mean terminal price : 107.92
5th percentile      : 68.77
95th percentile     : 158.87
95% 1-year VaR (loss): 31.23%


### ✏️ Your turn — probability of loss

Using `terminal_return` from the simulation above, compute `prob_loss` — the **fraction of paths that ended below the starting price** (i.e. a negative return). (Hint: a mask, then `.mean()`.)

In [ ]:
prob_loss = None   # TODO: fraction of paths with terminal_return < 0

In [ ]:
assert prob_loss is not None, "Replace None with your code."
assert 0.0 <= prob_loss <= 1.0, "A probability must be in [0, 1]."
assert np.isclose(prob_loss, (terminal_return < 0).mean()), "Not quite - count negatives / total."
print(f"✅ Correct! P(loss over 1 year) = {prob_loss:.1%}")

## 10. Linear algebra for portfolios

A portfolio is a **weighted combination of assets**, and its risk depends on how the assets **move together**. Two pieces of linear algebra capture this — and the `@` operator does matrix multiplication.

- **Portfolio return series:** `R @ w` — multiply the `(days, assets)` return matrix by the weight vector to get one portfolio return per day.
- **Covariance matrix `Σ`:** `np.cov(R, rowvar=False)` measures how each pair of assets co-moves (`rowvar=False` says "columns are the variables/assets").
- **Portfolio variance:** `w @ Σ @ w` — and its square root is the portfolio volatility.

In [43]:
rng4 = np.random.default_rng(2024)
# Three assets with different drifts/vols (so the example isn't degenerate):
R = rng4.normal([0.0003, 0.0005, 0.0002], [0.008, 0.014, 0.006], size=(252, 3))
w = np.array([0.5, 0.3, 0.2])             # portfolio weights, summing to 1

port_returns = R @ w                       # (252,3) @ (3,) -> (252,) one return per day
print("portfolio weights sum to:", w.sum())
print("portfolio annualized return:", round(port_returns.mean() * 252, 4))

cov = np.cov(R, rowvar=False)              # (3,3) daily covariance matrix
corr = np.corrcoef(R, rowvar=False)        # (3,3) correlation matrix
print("\ncorrelation matrix:\n", corr.round(2))

portfolio weights sum to: 1.0
portfolio annualized return: 0.1599

correlation matrix:
 [[ 1.    0.02 -0.05]
 [ 0.02  1.    0.  ]
 [-0.05  0.    1.  ]]


Now the punchline of diversification: combining assets that aren't perfectly correlated gives a portfolio volatility that's **lower** than the weighted average of the individual volatilities. The matrix form `w @ cov @ w` computes the portfolio variance directly.

In [44]:
port_vol_annual = np.sqrt(w @ cov @ w) * np.sqrt(252)
weighted_avg_vol = w @ (R.std(axis=0) * np.sqrt(252))
print("portfolio annualized vol      :", round(port_vol_annual, 4))
print("weighted-avg individual vol   :", round(weighted_avg_vol, 4), "<- higher: diversification helped")

portfolio annualized vol      : 0.0955
weighted-avg individual vol   : 0.1503 <- higher: diversification helped


### ✏️ Your turn — portfolio volatility

Using `cov` and weights `w` above, compute `port_vol` — the **annualized** portfolio volatility: $\sqrt{w^\top \Sigma\, w}\times\sqrt{252}$.

In [ ]:
port_vol = None   # TODO: annualized portfolio volatility from cov and w

In [ ]:
assert port_vol is not None, "Replace None with your code."
ref = np.sqrt(w @ cov @ w) * np.sqrt(252)
assert np.isclose(port_vol, ref), f"Expected {ref:.4f}, got {port_vol}"
weighted_avg = w @ (R.std(axis=0) * np.sqrt(252))
assert port_vol < weighted_avg, "Diversified vol should be below the weighted-average vol."
print(f"✅ Correct! portfolio vol = {port_vol:.4f} (< weighted-avg {weighted_avg:.4f}: diversification!)")

## 11. The vectorization mindset — rank a whole universe at once

Putting the habit to work: instead of looping over assets, compute a metric for *all* of them in one line, then rank. **`argsort`** is the key tool — it returns the **indices** that would sort an array (ascending), which is exactly what you need to pick the top N.

In [47]:
# 50 assets, one year of daily returns each:
universe = np.random.default_rng(11).normal(0.0003, 0.011, size=(252, 50))

# Every asset's Sharpe in ONE line (axis=0 reduces over days, per asset):
sharpes = universe.mean(axis=0) / universe.std(axis=0) * np.sqrt(252)
print("computed", sharpes.size, "Sharpe ratios, zero loops")
print("argsort gives ascending indices; last one = best:", sharpes.argsort()[-1])
print("best Sharpe:", round(sharpes.max(), 2))

computed 50 Sharpe ratios, zero loops
argsort gives ascending indices; last one = best: 21
best Sharpe: 2.18


To get the **top 5 highest**, take the *last* 5 of the ascending `argsort` and reverse them so the best comes first.

In [48]:
top5 = np.argsort(sharpes)[-5:][::-1]
print("top-5 asset indices (best first):", top5)
print("their Sharpes                   :", sharpes[top5].round(2))

top-5 asset indices (best first): [21 34 13 22 16]
their Sharpes                   : [2.18 1.94 1.75 1.68 1.46]


### ✏️ Your turn — top-5 by Sharpe

From `sharpes` above, build `top5` — the indices of the **5 highest-Sharpe assets, best first**. (Hint: `np.argsort` is ascending; slice the end and reverse with `[::-1]`.)

In [ ]:
top5 = None   # TODO: indices of the 5 highest-Sharpe assets, highest first

In [ ]:
assert top5 is not None, "Replace None with your code."
top5 = np.asarray(top5)
assert top5.shape == (5,), f"Expected 5 indices, got shape {top5.shape}"
assert top5[0] == sharpes.argmax(), "First element should be the best asset."
vals = sharpes[top5]
assert np.all(np.diff(vals) <= 0), "Should be sorted highest -> lowest."
print("✅ Correct! top-5 asset indices:", top5, " Sharpes:", np.round(vals, 2))

## \U0001f3c1 Mini-challenge: put it together

Bundle the workflow into a reusable function — exactly the kind of helper you'll paste into research notebooks all year.

### ✏️ Your turn — a Sharpe function

Implement `annualized_sharpe(prices)` that takes a 1-D price array and returns the **annualized Sharpe ratio** (assume risk-free rate = 0). Steps: prices → simple returns (`p[1:]/p[:-1] - 1`) → `mean/std × √252`.

In [ ]:
def annualized_sharpe(prices):
    # TODO: return the annualized Sharpe ratio (rf = 0)
    return None

In [ ]:
test_px = 100 * np.cumprod(1 + np.random.default_rng(5).normal(0.0005, 0.01, 500))
got = annualized_sharpe(test_px)
assert got is not None, "Return a value, not None."
_r = test_px[1:] / test_px[:-1] - 1
expected = _r.mean() / _r.std() * np.sqrt(252)
assert np.isclose(got, expected), f"Expected {expected:.4f}, got {got}"
print(f"✅ Correct! annualized Sharpe of the test series = {got:.2f}")

## Cheat sheet

| Task | NumPy |
|---|---|
| Create | `np.array`, `np.arange`, `np.linspace`, `np.zeros`, `np.ones`, `np.full` |
| Random (seeded) | `rng = np.random.default_rng(seed)`; `rng.normal(...)` |
| Inspect / reshape | `a.shape`, `a.ndim`, `a.size`, `a.dtype`, `a.reshape(...)` |
| Slice | `a[1:4]`, `a[:, j]` (column), `a[i, :]` (row), `.copy()` for independence |
| Mask / select | `a[a > 0]`, `(c1) & (c2)`, `np.where(cond, x, y)`, `mask.mean()` |
| Prices→returns | `p[1:]/p[:-1]-1` (simple); `np.diff(np.log(p))` (log) |
| Aggregate | `a.mean(axis=0)` per-asset, `axis=1` per-day, none = scalar |
| Cumulative | `np.cumsum`, `np.cumprod`, `np.maximum.accumulate` (running peak) |
| Portfolio | `R @ w`, `w @ cov @ w`, `np.cov(R, rowvar=False)` |
| Rank | `argmax`, `argsort` (return **indices**) |

**Golden rule:** writing a `for` loop over prices or assets? Ask whether a vectorized op does it instead. Usually it does.

## Stretch goals (bring questions to the next meeting)

1. Add a **risk-free rate** argument to `annualized_sharpe`.
2. Build a **2-asset efficient frontier**: sweep weight of asset A over `np.linspace(0, 1, 50)` and find the minimum-variance mix.
3. Re-run the Monte Carlo with `sigma = 0.40` — how does `prob_loss` change, and why?
4. Try a **20-day moving average** with pure NumPy slicing. Notice it's painful — that's why **Pandas** (Notebook 02) exists.

---

### What's next
**Notebook 02 — Pandas for Financial Data:** labeled, time-indexed data on top of these arrays (`DataFrame`, `DatetimeIndex`, `resample`, `rolling`, `pct_change`, `groupby`) — the bridge into QuantConnect's research environment.

**NumPy docs:** [Beginners guide](https://numpy.org/doc/stable/user/absolute_beginners.html) · [Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html) · [Random generator](https://numpy.org/doc/stable/reference/random/generator.html)

*MAT Education · Foundations · Module 1.*